In [1]:
# SPX vol surface study
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

_nb = globals().get("__vsc_ipynb_file__")
ROOT = None
for start in ([Path(_nb)] if _nb else []) + [Path.cwd()]:
    p = start.resolve()
    if p.suffix == ".ipynb":
        p = p.parent
    for candidate in (p, *p.parents):
        if (candidate / "vol_surface.py").is_file():
            ROOT = candidate
            break
    if ROOT is not None:
        break
if ROOT is None:
    raise FileNotFoundError(
        "Open Signal_Monitor in Cursor. "
        f"Could not find vol_surface.py from notebook={_nb!r} cwd={Path.cwd()!r}"
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from vol_surface import (
    VolSurfaceConfig,
    VolSurfaceStudy,
    ensure_runtime_deps,
    load_deepseek_api_key,
)

ensure_runtime_deps()
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update(plt.rcParamsDefault)

cfg = VolSurfaceConfig(
    underlying="US..SPX",
    max_dte=60,
    lookback_days=5,
    use_deepseek=bool(load_deepseek_api_key(ROOT / ".env")),
    cache_dir=ROOT / "research" / "data" / "vol_surface",
)
study = VolSurfaceStudy(cfg)

study.fetch_live()
study.load_history()


2026-06-27 18:16:45,910 | 6480 | 6296 | [open_context_base.py:411] _init_connect_sync: New connect ready: conn=7476579269228745086(1) context=<futu.quote.open_quote_context.OpenQuoteContext object at 0x0000021CA92B9FA0>


Snapshot batch failed: Get Market Snapshot request failed due to high frequency. Maximum 60 times per 30 seconds.
Snapshot batch failed: Get Market Snapshot request failed due to high frequency. Maximum 60 times per 30 seconds.
Snapshot batch failed: Get Market Snapshot request failed due to high frequency. Maximum 60 times per 30 seconds.


2026-06-27 18:17:30,797 | 6480 | 20284 | [open_context_base.py:521] on_disconnect: Disconnected: conn=0(1) reason=CallClose


2026-06-27 18:18:09,936 | 6480 | 20284 | [open_context_base.py:521] on_disconnect: Disconnected: conn=0(2) reason=CallClose


In [2]:
from IPython.display import clear_output, display

full = False
local = True

clear_output(wait=True)
display(study.plot_delta_surface(full=full, local=local))


FigureWidget({
    'data': [{'cmax': 63.20822919912117,
              'cmin': 0.0,
              'colorbar': {'len': 0.65, 'thickness': 18, 'title': {'text': 'Local vol (%)'}},
              'colorscale': [[0.0, '#000004'], [0.1111111111111111, '#180f3d'],
                             [0.2222222222222222, '#440f76'], [0.3333333333333333,
                             '#721f81'], [0.4444444444444444, '#9e2f7f'],
                             [0.5555555555555556, '#cd4071'], [0.6666666666666666,
                             '#f1605d'], [0.7777777777777778, '#fd9668'],
                             [0.8888888888888888, '#feca8d'], [1.0, '#fcfdbf']],
              'name': 'Local vol',
              'showscale': True,
              'type': 'surface',
              'uid': 'a10540ba-9968-4b5e-b7e3-a1baf05a52c1',
              'x': {'bdata': ('mpmZmZmZ4b8iIiIiIiLgv1ZVVVVVVd' ... 'VVVVXdPyIiIiIiIuA/mpmZmZmZ4T8='),
                    'dtype': 'f8'},
              'y': {'bdata': 'AwQFBgoLDA0OERITFB

In [3]:
study.print_conclusion(delta_lo=-0.05, delta_hi=0.05)

2026-06-27 18:18:06,228 | 6480 | 6296 | [open_context_base.py:411] _init_connect_sync: New connect ready: conn=7476579606107621077(2) context=<futu.quote.open_quote_context.OpenQuoteContext object at 0x0000021CAD4CA5A0>
- Vol level is sharply higher: mean IV in the -0.05 to +0.05 delta band is 18.5%, up 3.7 vol points versus the prior four-session average, and VIX at 18.4% is 2.0 points above its five-day level.
- Put/call skew has widened dramatically to +14.0 vol points in the band, compared to zero spread in each of the prior four sessions — puts are now significantly richer than calls.
- Sentiment reflects fear: downside protection is being bid, while call activity is mixed and selective; there is no dominant event hump in the term structure.
- Local-vol diagnostics show 29 ML IV outliers but zero local-vol spikes, suggesting the outlier quotes may be stale or tied to specific event pockets rather than a broad local-vol anomaly.
